# Language-Aware Router

## 1. Project Overview

**Task:** Detect the language of incoming text, then route it to the correct NLP pipeline — sentiment analysis, translation, summarization — based on language and task.

**Stack:** `LangChain` + `ChatOllama` + `qwen3.5:9b` for LLM processing, heuristic + LLM for language detection.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Detect** language of incoming text |
| 2 | **Route** text to appropriate language-specific pipelines |
| 3 | **Handle** unknown languages with fallback |
| 4 | **Build** a complete multilingual NLP router |

## 3. Why This Matters

- **Global applications** receive input in many languages
- **Different languages** may need different models or prompts
- **Routing** prevents processing text with the wrong model
- **Fallback handling** ensures graceful degradation

## 4. Setup

In [ ]:
import os, json, re, warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

LLM_MODEL = "qwen3.5:9b"
llm = ChatOllama(model=LLM_MODEL, temperature=0)

def clean(text):
    if "<think>" in text: text = text.split("</think>")[-1].strip()
    return text.strip()

def ask(system, user):
    return clean(llm.invoke([SystemMessage(content=system), HumanMessage(content=user)]).content)

print(f"LLM ready: {ask('Reply with one word.', 'Say ready.')[:20]}")

## 5. Language Detection

In [ ]:
# Heuristic language detection using common words
LANG_MARKERS = {
    "en": ["the", "is", "are", "was", "have", "this", "that", "with", "for"],
    "es": ["el", "la", "los", "las", "es", "son", "una", "para", "por", "que"],
    "fr": ["le", "la", "les", "des", "est", "une", "pour", "dans", "que", "sont"],
    "de": ["der", "die", "das", "ist", "ein", "eine", "und", "nicht", "mit"],
    "it": ["il", "la", "che", "non", "per", "una", "sono", "della", "questo"],
    "pt": ["o", "que", "de", "um", "para", "com", "uma", "por", "mais"],
}

def detect_language_heuristic(text):
    words = text.lower().split()
    scores = {}
    for lang, markers in LANG_MARKERS.items():
        scores[lang] = sum(1 for w in words if w in markers)
    best = max(scores, key=scores.get)
    return best if scores[best] >= 2 else "unknown"

# LLM-based detection for uncertain cases
DETECT_SYSTEM = """Detect the language of this text. Return JSON:
{"language_code": "en|es|fr|de|it|pt|other", "language_name": "English|Spanish|...", "confidence": 0.0-1.0} /no_think"""

def detect_language_llm(text):
    resp = ask(DETECT_SYSTEM, f"Text: {text}")
    r = resp
    if "```" in r: r = re.sub(r"```(?:json)?\s*", "", r).replace("```", "")
    s, e = r.find("{"), r.rfind("}") + 1
    if s >= 0 and e > s:
        try: return json.loads(r[s:e])
        except: pass
    return {"language_code": "unknown", "language_name": "Unknown", "confidence": 0.0}

def detect_language(text):
    heuristic = detect_language_heuristic(text)
    if heuristic != "unknown":
        return {"language_code": heuristic, "method": "heuristic"}
    llm_result = detect_language_llm(text)
    llm_result["method"] = "llm"
    return llm_result

# Test detection
test_texts = [
    "I love this product, it works perfectly!",
    "Me encanta este producto, funciona muy bien!",
    "J'adore ce produit, il fonctionne parfaitement!",
    "Ich liebe dieses Produkt, es funktioniert perfekt!",
    "Bu urun harika, cok memnunum!",
]

print("LANGUAGE DETECTION")
print("=" * 60)
for text in test_texts:
    result = detect_language(text)
    print(f"  [{result.get('language_code','?'):<3}] ({result.get('method','?'):<10}) {text[:50]}...")

---
## 6. NLP Pipelines

In [ ]:
# Define language-specific processing pipelines
def sentiment_pipeline(text, lang):
    resp = ask(f"Classify sentiment as positive/negative/neutral. Respond in {lang}. Return JSON: {{\"sentiment\":\"...\",\"confidence\":0.0-1.0}} /no_think",
        f"Text: {text}")
    return {"task": "sentiment", "result": resp}

def summarize_pipeline(text, lang):
    resp = ask(f"Summarize in 1 sentence in {lang}. /no_think", f"Text: {text}")
    return {"task": "summary", "result": resp}

def translate_pipeline(text, target_lang="English"):
    resp = ask(f"Translate to {target_lang}. Return ONLY the translation. /no_think", text)
    return {"task": "translate", "result": resp}

PIPELINES = {
    "en": {"sentiment": lambda t: sentiment_pipeline(t, "English"),
           "summary": lambda t: summarize_pipeline(t, "English")},
    "es": {"sentiment": lambda t: sentiment_pipeline(t, "Spanish"),
           "translate": lambda t: translate_pipeline(t, "English")},
    "fr": {"sentiment": lambda t: sentiment_pipeline(t, "French"),
           "translate": lambda t: translate_pipeline(t, "English")},
    "de": {"translate": lambda t: translate_pipeline(t, "English")},
}

print("Registered pipelines:")
for lang, tasks in PIPELINES.items():
    print(f"  {lang}: {', '.join(tasks.keys())}")

## 7. Router Implementation

In [ ]:
def route(text, task="sentiment"):
    detection = detect_language(text)
    lang = detection.get("language_code", "unknown")

    if lang in PIPELINES and task in PIPELINES[lang]:
        result = PIPELINES[lang][task](text)
        return {"lang": lang, "task": task, "routed": True, "result": result}
    elif lang in PIPELINES and "translate" in PIPELINES[lang]:
        translated = translate_pipeline(text, "English")
        result = PIPELINES["en"].get(task, lambda t: {"task": task, "result": "unsupported"})(translated["result"])
        return {"lang": lang, "task": task, "routed": True, "via_translation": True, "result": result}
    elif lang != "unknown":
        translated = translate_pipeline(text, "English")
        result = PIPELINES["en"].get(task, lambda t: {"task": task, "result": "unsupported"})(translated["result"])
        return {"lang": lang, "task": task, "routed": True, "via_translation": True, "result": result}
    else:
        return {"lang": "unknown", "task": task, "routed": False, "result": {"task": task, "result": "Language not detected"}}

# Test routing
print("ROUTING DEMO")
print("=" * 70)
route_tests = [
    ("I love this product!", "sentiment"),
    ("Me encanta este producto!", "sentiment"),
    ("J'adore ce produit!", "sentiment"),
    ("Ich liebe dieses Produkt!", "sentiment"),
]

for text, task in route_tests:
    result = route(text, task)
    print(f"\n  [{result['lang']}] {text}")
    print(f"  Task: {task} | Routed: {result['routed']}")
    r = result.get("result", {})
    if isinstance(r, dict):
        print(f"  Result: {r.get('result', str(r))[:80]}")

## 8. Batch Processing

In [ ]:
BATCH_TEXTS = [
    "This conference was incredibly insightful and well organized.",
    "Esta conferencia fue increiblemente perspicaz y bien organizada.",
    "Cette conference etait incroyablement perspicace et bien organisee.",
    "Bu konferans inanilmaz derecede aydinlatici ve iyi organize edilmisti.",
]

print("BATCH ROUTING")
print("=" * 70)
for text in BATCH_TEXTS:
    detection = detect_language(text)
    lang = detection.get("language_code", "?")
    method = detection.get("method", "?")
    print(f"  [{lang}] ({method}) {text[:50]}...")

## 9. Error Handling & Fallback

In [ ]:
EDGE_CASES = [
    "",
    "123 456 789",
    "Hello world me encanta this product",
    "a",
]

print("EDGE CASES")
print("=" * 50)
for text in EDGE_CASES:
    detection = detect_language(text) if text else {"language_code": "empty", "method": "skip"}
    print(f"  [{detection.get('language_code','?'):<7}] '{text[:40]}'")

## 10. Common Mistakes & Key Takeaways

| Mistake | Fix |
|---------|-----|
| No language detection | Always detect before processing |
| Hardcoding language | Auto-detect and route dynamically |
| No fallback | Translate to English as fallback |
| Short text detection | Use LLM for ambiguous/short text |

| # | Takeaway |
|---|----------|
| 1 | **Heuristic detection** is fast for clear cases |
| 2 | **LLM fallback** handles ambiguous/short text |
| 3 | **Route-then-process** ensures correct model usage |
| 4 | **Translation fallback** enables processing any language |
| 5 | **Logging language distribution** helps optimize pipeline |

---
*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*

